# Calcium Trace Viewer

Plot calcium traces from a Minian output directory.

Produces two figures:

1. Full trace (vertically stacked per unit) with light-gray focus-zone boxes marking zoom regions, and a row of zoom panels below.
2. Max projection + filled colored spatial footprints (matching trace colors).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.patches import Rectangle
from matplotlib.ticker import FixedLocator, FuncFormatter, MultipleLocator

from placecell.visualization import plot_footprints_filled

## Config

- `MINIAN_DIR`: path to a Minian output dir (must contain `C.zarr`, `A.zarr`, `max_proj.zarr`).
- `FS_HZ`: sampling rate; set to `None` to plot against frame index.
- `N_ZOOMS` + `ZOOM_WIDTH_MIN`: auto-distribute N zoom panels of this width, evenly across the recording.
- `ZOOM_STARTS_MIN`: optional explicit list of start times (min). If non-empty, overrides `N_ZOOMS`.

Start times are snapped to the nearest 10-minute boundary so tick labels land at round values.

In [ ]:
MINIAN_DIR = Path("/Volumes/ProcData/minizero_long/full_preproc/filt/wlms/minian_int")

FS_HZ: float | None = 20.0

N_ZOOMS = 3
ZOOM_WIDTH_MIN = 20.0
ZOOM_STARTS_MIN: list[float] = []  # e.g. [0, 240, 450] to override N_ZOOMS

## Helpers

In [ ]:
def offset_traces(C: np.ndarray) -> tuple[np.ndarray, float]:
    """Return traces shifted by per-unit offsets for stacked plotting."""
    scale = np.nanpercentile(C, 99)
    if not np.isfinite(scale) or scale <= 0:
        scale = np.nanmax(np.abs(C)) or 1.0
    offsets = np.arange(C.shape[0]) * scale
    return C + offsets[:, None], float(scale)


def format_hm(seconds: float, _pos: int | None = None) -> str:
    """Format a time-in-seconds tick as 'Xh Ym', dropping 'Ym' when m==0."""
    s = max(0.0, float(seconds))
    h = int(s // 3600)
    m = int(round((s - h * 3600) / 60))
    if m == 60:
        h, m = h + 1, 0
    return f"{h}h" if m == 0 else f"{h}h {m}m"


def resolve_zoom_starts(
    explicit_starts: list[float],
    n_zooms: int,
    total_min: float,
    width_min: float,
) -> list[float]:
    """Return zoom-window start times (min), snapped to multiples of 10 min."""
    if explicit_starts:
        raw = list(explicit_starts)
    elif n_zooms > 0:
        usable = max(0.0, total_min - width_min)
        if n_zooms == 1:
            raw = [0.0]
        else:
            step = usable / (n_zooms - 1)
            raw = [i * step for i in range(n_zooms)]
    else:
        return []
    return [round(s / 10.0) * 10.0 for s in raw]

## Load calcium data

In [ ]:
ds_C = xr.open_zarr(MINIAN_DIR / "C.zarr", consolidated=False)
C = ds_C["C"].values  # (unit, frame)
unit_ids = ds_C["unit_id"].values
frames = ds_C["frame"].values
n_units, n_frames = C.shape

total_min = (n_frames / FS_HZ) / 60.0 if FS_HZ else 0.0
print(f"{n_units} units × {n_frames} frames  (~{total_min:.1f} min @ {FS_HZ} Hz)")

## Full trace + zoom panels

Zoom starts are resolved from either `ZOOM_STARTS_MIN` (if set) or `N_ZOOMS` (auto-distributed). Each zoom panel is `ZOOM_WIDTH_MIN` minutes wide with ticks at start / +10 min / +20 min.

In [ ]:
use_time = FS_HZ is not None
zoom_starts = resolve_zoom_starts(ZOOM_STARTS_MIN, N_ZOOMS, total_min, ZOOM_WIDTH_MIN)

if use_time:
    x_sec = frames / FS_HZ
    width_sec = ZOOM_WIDTH_MIN * 60.0
    zoom_starts_sec = [s * 60.0 for s in zoom_starts]
    zoom_windows = [
        (int(round(s * FS_HZ)), int(round((s + width_sec) * FS_HZ))) for s in zoom_starts_sec
    ]
else:
    x_sec = frames.astype(float)
    zoom_starts_sec = []
    zoom_windows = []

stacked, step = offset_traces(C)
yticks_all = np.arange(n_units) * step
ymin, ymax = -step, yticks_all[-1] + step

if n_units == 1:
    yticks = [yticks_all[0]]
    ytick_labels = [str(unit_ids[0])]
else:
    yticks = [yticks_all[0], yticks_all[-1]]
    ytick_labels = [str(unit_ids[0]), str(unit_ids[-1])]

cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
trace_colors = [cycle[i % len(cycle)] for i in range(n_units)]

n_zoom = max(1, len(zoom_windows))
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(2, n_zoom, height_ratios=[3, 2])

ax_full = fig.add_subplot(gs[0, :])
for i in range(n_units):
    ax_full.plot(x_sec, stacked[i], color=trace_colors[i], linewidth=0.4)
ax_full.set_yticks(yticks)
ax_full.set_yticklabels(ytick_labels)
ax_full.set_ylabel("Unit ID")
ax_full.set_xlabel("Time" if use_time else "Frame")
ax_full.set_xlim(x_sec[0], x_sec[-1])
ax_full.set_ylim(ymin, ymax)
if use_time:
    ax_full.xaxis.set_major_locator(MultipleLocator(3600))
    ax_full.xaxis.set_major_formatter(FuncFormatter(format_hm))
ax_full.set_title(f"Calcium traces — {MINIAN_DIR.name}/C.zarr ({n_units} units, {n_frames} frames)")

for k, (f0, f1) in enumerate(zoom_windows):
    f0 = max(0, int(f0))
    f1 = min(n_frames, int(f1))
    if f1 <= f0:
        continue
    x0, x1 = x_sec[f0], x_sec[f1 - 1]
    ax_full.add_patch(
        Rectangle(
            (x0, ymin), x1 - x0, ymax - ymin,
            facecolor="lightgray", edgecolor="none", alpha=0.6, zorder=0,
        )
    )

    ax_zoom = fig.add_subplot(gs[1, k])
    seg_x = x_sec[f0:f1]
    seg = stacked[:, f0:f1]
    for i in range(n_units):
        ax_zoom.plot(seg_x, seg[i], color=trace_colors[i], linewidth=0.6)
    ax_zoom.set_ylim(ymin, ymax)
    ax_zoom.set_xlabel("Time" if use_time else "Frame")
    if use_time:
        start_sec = zoom_starts_sec[k]
        end_sec = start_sec + ZOOM_WIDTH_MIN * 60.0
        ax_zoom.set_xlim(start_sec, end_sec)
        ax_zoom.xaxis.set_major_locator(
            FixedLocator([start_sec, start_sec + 600.0, start_sec + 1200.0])
        )
        ax_zoom.xaxis.set_major_formatter(FuncFormatter(format_hm))
    else:
        ax_zoom.set_xlim(seg_x[0], seg_x[-1])
    ax_zoom.set_title(f"Zoom {k + 1}: {format_hm(x0)}–{format_hm(x1)}")
    if k == 0:
        ax_zoom.set_yticks(yticks)
        ax_zoom.set_yticklabels(ytick_labels)
        ax_zoom.set_ylabel("Unit ID")
    else:
        ax_zoom.set_yticks([])

fig.tight_layout()
plt.show()

## Max projection + colored footprints

Same color order as the trace plot, so a unit's trace color matches its mask color.

In [ ]:
A = xr.open_zarr(MINIAN_DIR / "A.zarr", consolidated=False)["A"].sel(unit_id=list(unit_ids))
max_proj = xr.open_zarr(MINIAN_DIR / "max_proj.zarr", consolidated=False)["max_proj"].values

fig = plot_footprints_filled(max_proj, A)
plt.show()

## Optional: save figures

Uncomment to write both figures to PDF.

In [ ]:
# output_dir = project_root / "output"
# output_dir.mkdir(exist_ok=True)
# plt.figure(fig.number)  # ensure fig is current before saving
# fig.savefig(output_dir / "calcium_footprints.pdf", dpi=150)
# # Re-run the traces cell with plt.gcf() captured to save; or copy the code above
# # and replace plt.show() with fig.savefig(output_dir / 'calcium_traces.pdf', dpi=150).